In [1]:
import os
import gc
import math

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter, defaultdict
from tqdm.auto import tqdm


In [ ]:
# ── Cấu hình ──────────────────────────────────────────────────────────────────
CONFIG = {
    "emb_dim":        16,
    "n_layers":       2,
    "lr":             1e-3,
    "weight_decay":   1e-3,
    "batch_size":     16384,
    "epochs":         50,
    "patience":       5,
    "k":              10,
    "checkpoint_dir": "/kaggle/working",
    "train_dir":      "/kaggle/input/datasets/b22dckh121lvnth/data-mining/Data/config_id=baseline/train",
    "val_dir":        "/kaggle/input/datasets/b22dckh121lvnth/data-mining/Data/config_id=baseline/val",
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)


COLS_NEEDED = [
    "reviewer_id",
    "parent_asin",
    "bpr_role",             
    "bpr_positive_weight",  
    "reliability_score",
    "strength_score",
]

# Thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị: {device}")

if device.type == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM    : {vram_gb:.1f} GB")

train_files = sorted([
    os.path.join(CONFIG["train_dir"], f)
    for f in os.listdir(CONFIG["train_dir"])
    if f.endswith(".parquet")
])
print(f"File train: {len(train_files)}")


Thiết bị: cuda
VRAM    : 14.6 GB
File train: 665


In [3]:
df = pd.read_parquet(train_files[0])
print(df[:5])

  parent_asin                   reviewer_id  rating  \
0  B08XXJ1Z98  AEJR7KGPDPJQ7ZAH6DAT3PHSCQSQ     5.0   
1  B00000JBHP  AFX2NNV35HHN2JWQKSL7SAC2F5YQ     4.0   
2  B08XXJ1Z98  AGZMVVC4F5EBNNB6BZS4UHW7BQ7A     5.0   
3  B00000JBHP  AEXOJCEFYK5F7GRAGJDCIWOX22XQ     5.0   
4  B08YNT6MQN  AEC6YLLM3SYXLHIQF7MPMH56BYQQ     1.0   

                                         review_text  text_len   timestamp  \
0  We absolutely loved the backdrop for my sons 3...        18  1578349759   
1                                       Works Great.         2  1578614877   
2                                      Great product         2  1578009439   
3  These headphones go above and beyond my expect...        53  1580329770   
4  Not much to review, since I couldn't get the u...        88  1578007855   

   helpful_vote  verified_purchase ingest_date  \
0             2               True  2026-03-30   
1             0               True  2026-03-30   
2             0               True  2026-03-30   


In [3]:
# ── BƯỚC 1: Lọc K-Core ────────────────────────────────────────────────────────
print("\n── BƯỚC 1: Lọc K-Core ──")

user_pos_counts = Counter()
item_pos_counts = Counter()

for f in tqdm(train_files, desc="Đếm positive edges"):
    df_tmp = pd.read_parquet(f, columns=["reviewer_id", "parent_asin", "bpr_role"])
    pos_df = df_tmp[df_tmp["bpr_role"] == "positive"]
    user_pos_counts.update(pos_df["reviewer_id"])
    item_pos_counts.update(pos_df["parent_asin"])
    del df_tmp, pos_df
    gc.collect()

# Giữ lại user / item có ít nhất 2 positive edge
valid_users = {u for u, c in user_pos_counts.items() if c >= 2}
valid_items = {i for i, c in item_pos_counts.items() if c >= 2}
del user_pos_counts, item_pos_counts
gc.collect()

print(f"User >= 2 pos : {len(valid_users):,}")
print(f"Item >= 2 pos : {len(valid_items):,}")

# Xây ID map — sort để đảm bảo reproducible
user_keys = sorted(valid_users)
item_keys = sorted(valid_items)
user_map  = {u: i for i, u in enumerate(user_keys)}
item_map  = {it: i for i, it in enumerate(item_keys)}
n_user    = len(user_map)
n_item    = len(item_map)
del valid_users, valid_items
gc.collect()

print(f"n_user = {n_user:,} | n_item = {n_item:,}")

emb_size_gb = (n_user + n_item) * CONFIG["emb_dim"] * 4 / 1024**3
print(f"Embedding table: {emb_size_gb:.2f} GB")



── BƯỚC 1: Lọc K-Core ──


Đếm positive edges:   0%|          | 0/665 [00:00<?, ?it/s]

User >= 2 pos : 6,503,171
Item >= 2 pos : 950,246
n_user = 6,503,171 | n_item = 950,246
Embedding table: 0.44 GB


In [ ]:
# ── BƯỚC 2: Xây Adjacency Matrix & BPR Data ───────────────────────────────────
print("\n── BƯỚC 2: Build Adjacency Matrix & BPR Data ──")

rows_all, cols_all, data_all = [], [], []
train_pairs_list = []
hard_neg_dict = defaultdict(list)  # uid → [item_id, ...]
user_pos_set  = defaultdict(set)   # uid → {item_id, ...}

FILES_PER_CHUNK = 10

for chunk_start in tqdm(range(0, len(train_files), FILES_PER_CHUNK), desc="Xử lý chunks"):
    # Đọc một chunk file
    dfs = []
    for f in train_files[chunk_start : chunk_start + FILES_PER_CHUNK]:
        try:
            dfs.append(pd.read_parquet(f, columns=COLS_NEEDED))
        except Exception as e:
            print(f"  Bỏ qua {os.path.basename(f)}: {e}")
    if not dfs:
        continue

    df = pd.concat(dfs, ignore_index=True)
    del dfs


    users      = pd.Categorical(df["reviewer_id"], categories=user_keys).codes.astype(np.int32)
    items      = pd.Categorical(df["parent_asin"],  categories=item_keys).codes.astype(np.int32)
    valid_mask = (users >= 0) & (items >= 0)
    roles      = df["bpr_role"].values

    # ── Positive → adjacency matrix + BPR positive pairs ──
    pos_mask = valid_mask & (roles == "positive")
    if pos_mask.any():
        u_pos = users[pos_mask]
        i_pos = items[pos_mask]

        # Edge weight: reliability * (0.5 + 0.5 * strength)
        edge_w = (
            df["reliability_score"].values[pos_mask]
            * (0.5 + 0.5 * df["strength_score"].values[pos_mask])
        ).astype(np.float32)

        # Loại giá trị không hợp lệ
        valid_ew       = np.isfinite(edge_w) & (edge_w > 0)
        u_pos, i_pos, edge_w = u_pos[valid_ew], i_pos[valid_ew], edge_w[valid_ew]

        # Adjacency matrix đối xứng
        i_offset = i_pos + n_user
        rows_all.append(np.concatenate([u_pos, i_offset]))
        cols_all.append(np.concatenate([i_offset, u_pos]))
        data_all.append(np.concatenate([edge_w, edge_w]))

        # BPR pairs: [uid, iid, pos_weight]
        pos_w = df["bpr_positive_weight"].values[pos_mask][valid_ew].astype(np.float32)
        train_pairs_list.append(np.column_stack([u_pos, i_pos, pos_w]))

        # Lưu positive set để tránh sample nhầm khi lấy random neg
        for u, i in zip(u_pos.tolist(), i_pos.tolist()):
            user_pos_set[u].add(i)

    # ── Hard negative → BPR negative pool ──
    neg_mask = valid_mask & (roles == "hard_negative")
    if neg_mask.any():
        for u, i in zip(users[neg_mask].tolist(), items[neg_mask].tolist()):
            hard_neg_dict[u].append(int(i))

    # "excluded" → bỏ hoàn toàn

    del df, users, items, valid_mask, roles
    gc.collect()

# Gộp thành numpy array
train_pairs = np.vstack(train_pairs_list).astype(np.float32)
del train_pairs_list
gc.collect()
print(f"BPR positive pairs: {len(train_pairs):,}")

# ── Xây sparse adjacency matrix ──
print("Đang đúc sparse adjacency matrix...")
N = n_user + n_item
A = sp.csr_matrix(
    (np.concatenate(data_all),
     (np.concatenate(rows_all), np.concatenate(cols_all))),
    shape=(N, N),
    dtype=np.float32,
)
del rows_all, cols_all, data_all
gc.collect()

# Chuẩn hóa đối xứng: D^{-1/2} A D^{-1/2}
print("Chuẩn hóa D^{-1/2} A D^{-1/2}...")
deg          = np.array(A.sum(axis=1)).flatten()
deg_inv_sqrt = np.zeros_like(deg, dtype=np.float32)
deg_inv_sqrt[deg > 0] = 1.0 / np.sqrt(deg[deg > 0])
A_norm = sp.diags(deg_inv_sqrt).dot(A).dot(sp.diags(deg_inv_sqrt)).tocoo()
del A, deg, deg_inv_sqrt
gc.collect()

# Chuyển sang PyTorch sparse CSR
print("Chuyển sang PyTorch sparse CSR...")
indices = torch.stack([
    torch.LongTensor(A_norm.row),
    torch.LongTensor(A_norm.col),
])
A_torch = (
    torch.sparse_coo_tensor(indices, torch.FloatTensor(A_norm.data), size=A_norm.shape)
    .to_sparse_csr()
    .to(device)
)
del A_norm, indices
gc.collect()

print(f"Graph: shape={A_torch.shape} | nnz={A_torch._nnz():,}")



── BƯỚC 2: Build Adjacency Matrix & BPR Data ──


Xử lý chunks:   0%|          | 0/67 [00:00<?, ?it/s]

BPR positive pairs: 26,694,636
Đang đúc sparse adjacency matrix...
Chuẩn hóa D^{-1/2} A D^{-1/2}...
Chuyển sang PyTorch sparse CSR...


/tmp/ipykernel_186/484544300.py:108: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  .to_sparse_csr()


Graph: shape=torch.Size([7453417, 7453417]) | nnz=52,231,236


In [6]:
# ── BƯỚC 3: Load Validation Data ─────────────────────────────────────────────
print("\n── BƯỚC 3: Load Validation Data ──")

val_files = sorted([
    os.path.join(CONFIG["val_dir"], f)
    for f in os.listdir(CONFIG["val_dir"])
    if f.endswith(".parquet")
])
print(f"File val: {len(val_files)}")

VAL_COLS       = ["reviewer_id", "parent_asin", "bpr_role", "bpr_positive_weight"]
val_pairs_list = []

for f in tqdm(val_files, desc="Đọc validation"):
    try:
        df_val = pd.read_parquet(f, columns=VAL_COLS)
    except Exception as e:
        print(f"  Bỏ qua {os.path.basename(f)}: {e}")
        continue

    users       = pd.Categorical(df_val["reviewer_id"], categories=user_keys).codes.astype(np.int32)
    items       = pd.Categorical(df_val["parent_asin"],  categories=item_keys).codes.astype(np.int32)
    pos_weights = df_val["bpr_positive_weight"].values
    
    valid_mask = (
        (users >= 0)
        & (items >= 0)
        & (df_val["bpr_role"].values == "positive")
        & ((pos_weights == 1.0) | (pos_weights == 0.5))
    )

    if valid_mask.any():
        w_val = pos_weights[valid_mask].astype(np.float32)
        val_pairs_list.append(np.column_stack([users[valid_mask], items[valid_mask], w_val]))

    del df_val
    gc.collect()

val_pairs = np.vstack(val_pairs_list).astype(np.float32)
del val_pairs_list
gc.collect()

print(f"Val pairs (warm + weight 1.0/0.5): {len(val_pairs):,}")

unique_w, cnt_w = np.unique(val_pairs[:, 2], return_counts=True)
for w, c in zip(unique_w, cnt_w):
    print(f"  weight={w:.1f}: {c:,} ({c / len(val_pairs) * 100:.1f}%)")



── BƯỚC 3: Load Validation Data ──
File val: 8


Đọc validation:   0%|          | 0/8 [00:00<?, ?it/s]

Val pairs (warm + weight 1.0/0.5): 6,523
  weight=0.5: 721 (11.1%)
  weight=1.0: 5,802 (88.9%)


In [7]:
# ── Model: LightGCN ───────────────────────────────────────────────────────────
class LightGCN(nn.Module):
    def __init__(self, n_users: int, n_items: int, emb_dim: int = 32, n_layers: int = 2):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers
        self.embedding = nn.Embedding(n_users + n_items, emb_dim)
        nn.init.xavier_uniform_(self.embedding.weight)

    def forward(self, A: torch.Tensor) -> torch.Tensor:
        E        = self.embedding.weight
        emb_list = [E]
        for _ in range(self.n_layers):
            E = torch.sparse.mm(A, E)
            emb_list.append(E)
        return torch.stack(emb_list, dim=0).mean(dim=0)

    def get_user_item_emb(self, A: torch.Tensor):
        E_star = self.forward(A)
        return E_star[:self.n_users], E_star[self.n_users:]


In [8]:
# ── Hàm hỗ trợ: negative sampling & loss ─────────────────────────────────────

def sample_negatives(
    user_ids: np.ndarray,
    hard_neg_dict: dict,
    user_pos_set: dict,
    n_items: int,
) -> torch.LongTensor:
    """
    Sample 1 negative item cho mỗi user trong batch.
      - 10% xác suất: chọn từ hard_negative pool (nếu user có)
      - 90% xác suất: random uniform, tránh trùng positive set
    """
    neg_ids = np.random.randint(0, n_items, size=len(user_ids))

    for idx, uid in enumerate(user_ids.tolist()):
        hard_neg = hard_neg_dict.get(uid, [])
        pos_set  = user_pos_set[uid]

        if hard_neg and np.random.rand() < 0.1:
            neg_ids[idx] = np.random.choice(hard_neg)
        else:
            neg = neg_ids[idx]
            while neg in pos_set:
                neg = np.random.randint(n_items)
            neg_ids[idx] = neg

    return torch.LongTensor(neg_ids)


def weighted_bpr_loss(
    u_emb: torch.Tensor,
    p_emb: torch.Tensor,
    n_emb: torch.Tensor,
    pos_weights: torch.Tensor,
    weight_decay: float,
) -> torch.Tensor:
    
    pos_scores = (u_emb * p_emb).sum(dim=-1)
    neg_scores = (u_emb * n_emb).sum(dim=-1)

    bpr_loss = -(pos_weights * F.logsigmoid(pos_scores - neg_scores)).mean()

    l2_reg = (
        u_emb.norm(2).pow(2)
        + p_emb.norm(2).pow(2)
        + n_emb.norm(2).pow(2)
    ) / u_emb.size(0)

    return bpr_loss + weight_decay * l2_reg


In [ ]:


def train_one_epoch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    A_torch: torch.Tensor,
    train_pairs: np.ndarray,
    hard_neg_dict: dict,
    user_pos_set: dict,
    config: dict,
    device: torch.device,
) -> float:
    model.train()

    pairs      = train_pairs[np.random.permutation(len(train_pairs))]
    bs         = config["batch_size"]
    total_loss = 0.0

    for start in tqdm(range(0, len(pairs), bs), desc="  Batches", leave=False):
        batch   = pairs[start : start + bs]
        uids    = batch[:, 0].astype(np.int32)
        pids    = batch[:, 1].astype(np.int32)
        weights = batch[:, 2]

        nids = sample_negatives(uids, hard_neg_dict, user_pos_set, model.n_items)

        uids_t    = torch.LongTensor(uids).to(device)
        pids_t    = torch.LongTensor(pids).to(device)
        nids_t    = nids.to(device)
        weights_t = torch.FloatTensor(weights).to(device)

        E_star = model(A_torch)
        u_e    = E_star[:model.n_users][uids_t]
        p_e    = E_star[model.n_users:][pids_t]
        n_e    = E_star[model.n_users:][nids_t]

        optimizer.zero_grad()
        loss = weighted_bpr_loss(u_e, p_e, n_e, weights_t, config["weight_decay"])
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(len(pairs) // bs, 1)


def evaluate_val_loss(
    model: nn.Module,
    A_torch: torch.Tensor,
    val_pairs_np: np.ndarray,
    hard_neg_dict: dict,
    user_pos_set: dict,
    config: dict,
    device: torch.device,
) -> float:
    """
    Tính weighted BPR loss trên tập val (không cập nhật gradient).
    E* được tính 1 lần trước vòng lặp batch để tiết kiệm thời gian.
    """
    model.eval()

    bs         = config["batch_size"]
    total_loss = 0.0
    n_batches  = 0


    with torch.no_grad():
        E_star = model(A_torch)

    pairs = val_pairs_np[np.random.permutation(len(val_pairs_np))]

    for start in tqdm(range(0, len(pairs), bs), desc="  Val batches", leave=False):
        batch   = pairs[start : start + bs]
        uids    = batch[:, 0].astype(np.int32)
        pids    = batch[:, 1].astype(np.int32)
        weights = batch[:, 2]

        nids = sample_negatives(uids, hard_neg_dict, user_pos_set, model.n_items)

        uids_t    = torch.LongTensor(uids).to(device)
        pids_t    = torch.LongTensor(pids).to(device)
        nids_t    = nids.to(device)
        weights_t = torch.FloatTensor(weights).to(device)

        with torch.no_grad():
            u_e  = E_star[:model.n_users][uids_t]
            p_e  = E_star[model.n_users:][pids_t]
            n_e  = E_star[model.n_users:][nids_t]
            loss = weighted_bpr_loss(u_e, p_e, n_e, weights_t, config["weight_decay"])

        total_loss += loss.item()
        n_batches  += 1

    avg_loss = total_loss / max(n_batches, 1)
    print(f"  Val Loss = {avg_loss:.4f}")
    return avg_loss


In [10]:
# ── BƯỚC 4: Training ──────────────────────────────────────────────────────────
print("\n── BƯỚC 4: Training ──")

model     = LightGCN(n_user, n_item, CONFIG["emb_dim"], CONFIG["n_layers"]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

best_val_loss = float("inf")
patience_cnt  = 0

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss = train_one_epoch(
        model, optimizer, A_torch,
        train_pairs, hard_neg_dict, user_pos_set,
        CONFIG, device,
    )
    val_loss = evaluate_val_loss(
        model, A_torch, val_pairs,
        hard_neg_dict, user_pos_set,
        CONFIG, device,
    )

    print(f"Epoch {epoch:03d} | Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}", end="")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_cnt  = 0
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss":           train_loss,
            "val_loss":             best_val_loss,
            "config":               CONFIG,
            "n_user":               n_user,
            "n_item":               n_item,
        }, f"{CONFIG['checkpoint_dir']}/best.pt")
        print(" ✓ checkpoint saved")
    else:
        patience_cnt += 1
        print(f" (patience {patience_cnt}/{CONFIG['patience']})")
        if patience_cnt >= CONFIG["patience"]:
            print(f"Early stopping tại epoch {epoch}.")
            break

print(f"\nBest Val Loss: {best_val_loss:.4f}")



── BƯỚC 4: Training ──
Model params: 119,254,672


  Batches:   0%|          | 0/1630 [00:00<?, ?it/s]

  Val batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Val Loss = 0.4924
Epoch 001 | Train Loss = 0.4083 | Val Loss = 0.4924 ✓ checkpoint saved


  Batches:   0%|          | 0/1630 [00:00<?, ?it/s]

...

  Val batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Val Loss = 0.5214
Epoch 006 | Train Loss = 0.2564 | Val Loss = 0.5214 (patience 4/5)


  Batches:   0%|          | 0/1630 [00:00<?, ?it/s]

  Val batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Val Loss = 0.5213
Epoch 007 | Train Loss = 0.2476 | Val Loss = 0.5213 (patience 5/5)
Early stopping tại epoch 7.

Best Val Loss: 0.4905


In [11]:
# ── BƯỚC 5: Export Embeddings ─────────────────────────────────────────────────
print("\n── BƯỚC 5: Export Embeddings ──")

ckpt = torch.load(f"{CONFIG['checkpoint_dir']}/best.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

with torch.no_grad():
    E_star   = model(A_torch)
    user_emb = E_star[:model.n_users].cpu()
    item_emb = E_star[model.n_users:].cpu()

torch.save(user_emb, f"{CONFIG['checkpoint_dir']}/user_emb.pt")
torch.save(item_emb, f"{CONFIG['checkpoint_dir']}/item_emb.pt")

assert user_emb.shape == (n_user, CONFIG["emb_dim"]), f"user_emb shape sai: {user_emb.shape}"
assert item_emb.shape == (n_item, CONFIG["emb_dim"]), f"item_emb shape sai: {item_emb.shape}"

print(f"user_emb : {tuple(user_emb.shape)}  ✓")
print(f"item_emb : {tuple(item_emb.shape)}  ✓")
print(f"Epoch    : {ckpt['epoch']} | Val Loss = {ckpt['val_loss']:.4f}")
print(f"Saved to : {CONFIG['checkpoint_dir']}/")



── BƯỚC 5: Export Embeddings ──
user_emb : (6503171, 16)  ✓
item_emb : (950246, 16)  ✓
Epoch    : 2 | Val Loss = 0.4905
Saved to : /kaggle/working/


In [13]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.0 MB/s eta 0:00:00:00:0100:01


In [2]:
import pandas as pd

In [ ]:
# ── BƯỚC 6: Sinh Candidates bằng FAISS LSH ────────────────────────────────────
import faiss


def generate_candidates_faiss_lsh(
    user_emb_path: str,
    item_emb_path: str,
    user_keys: list,
    item_keys: list,
    user_history_dict: dict,
    top_k: int = 200,
    n_bits: int = 256,
    search_batch_size: int = 10000,  # search từng batch user
) -> pd.DataFrame:

    user_emb = torch.load(user_emb_path).cpu().numpy().astype(np.float32)
    item_emb = torch.load(item_emb_path).cpu().numpy().astype(np.float32)

    index = faiss.IndexLSH(item_emb.shape[1], n_bits)
    index.add(item_emb)

    results = []
    n_users = user_emb.shape[0]

    for batch_start in tqdm(range(0, n_users, search_batch_size), desc="Sinh candidates"):
        batch_emb = user_emb[batch_start : batch_start + search_batch_size]
        distances, indices = index.search(batch_emb, top_k + 50)

        for local_idx, (raw_items, raw_scores) in enumerate(zip(indices, distances)):
            u_idx   = batch_start + local_idx
            history = user_history_dict.get(u_idx, set())

            clean_items, clean_scores = [], []
            for i_idx, score in zip(raw_items, raw_scores):
                if i_idx == -1 or i_idx in history:
                    continue
                clean_items.append(item_keys[i_idx])
                clean_scores.append(float(score))
                if len(clean_items) == top_k:
                    break

            results.append({
                "reviewer_id":     user_keys[u_idx],
                "candidate_asins": clean_items,
                "graph_scores":    clean_scores,
            })

    df = pd.DataFrame(results).explode(["candidate_asins", "graph_scores"])
    df = df.rename(columns={"candidate_asins": "parent_asin", "graph_scores": "graph_score"})
    df["graph_score"] = df["graph_score"].astype(np.float32)
    df["graph_rank"]  = df.groupby("reviewer_id").cumcount().add(1).astype(np.int32)
    return df


candidates_df = generate_candidates_faiss_lsh(
    user_emb_path      = f"{CONFIG['checkpoint_dir']}/user_emb.pt",
    item_emb_path      = f"{CONFIG['checkpoint_dir']}/item_emb.pt",
    user_keys          = user_keys,
    item_keys          = item_keys,
    user_history_dict  = user_pos_set,
    top_k              = 200,
    n_bits             = 256,
)

print(candidates_df.head())
candidates_df.to_parquet("graph_candidates_final.parquet", index=False)
print(f"Đã lưu {len(candidates_df):,} dòng → graph_candidates_final.parquet")


Sinh candidates:   0%|          | 0/651 [00:00<?, ?it/s]